# 🏠 House Price Prediction with Neural Networks — Improved 

This is the **second part** of our lecture. In the previous notebook we built a complete, working neural network pipeline. It works — but can we do better?

In this notebook we keep the **same structure** and apply **5 targeted tricks** to squeeze out more performance. Each trick targets a specific weakness of the baseline approach.

| # | Trick | What it does |
|---|-------|-------------|
| 🔧 1 | **Log-transform the target** | The model learns *relative* errors instead of absolute dollar errors |
| 🔧 2 | **Feature engineering** | We give the network pre-computed interactions that are hard to learn from raw features |
| 🔧 3 | **Mini-batch training (DataLoader)** | Faster convergence and better generalization with stochastic gradient estimates |
| 🔧 4 | **Learning Rate Scheduler** | Automatically reduces the learning rate when training plateaus |
| 🔧 5 | **Kaiming weight initialization** | Smarter starting weights matched to ReLU activations |

## 🎯 Your tasks in this notebook

| # | Where | What to do |
|---|-------|-----------|
| 🎯 1 | Section 2 — Feature Engineering | Complete the three engineered feature formulas |
| 🎯 2 | Section 2 — Separate features | Compute the log-transformed prices |
| 🎯 3 | Section 2 — DataLoader | Create a TensorDataset and a DataLoader |
| 🎯 4 | Section 4 — Scheduler | Fill in the ReduceLROnPlateau parameters |

---

## 📦 Step 0: Install Required Packages

Same packages as the base notebook.

> ⚠️ **After running this cell, restart the runtime:** go to **Runtime → Restart runtime**, then re-run all cells from the top.

In [ ]:
# ============================================
# INSTALLATION — Run this first!
# ============================================

%pip install torch scikit-learn kagglehub pandas numpy matplotlib seaborn --quiet

!wget https://raw.githubusercontent.com/eth-bmai-fs26/coding-exercises/week1/cx3-intro-to-nns/utils.py

print("✅ All packages installed successfully!")

## 📦 Step 1: Import Libraries

Same setup as the base notebook — we import everything from `utils.py`, which already includes all the libraries (`TensorDataset` and `DataLoader` for Trick 3 are in there too) and the helper functions.

> 💡 You can open `utils.py` at any time to see exactly what each helper function does.

In [ ]:
# Import all libraries and helper functions from our utils file
from utils import *



print("✅ Everything imported successfully!")

---

# 📊 1. Data Loading & Exploration

This section is **identical** to the base notebook — same dataset, same exploration. We use `load_dataset()` to download and clean the data in one step, and `plot_dataset()` to visualise the four key charts.

> 💡 If you haven't gone through the base notebook yet, take a quick look at the charts and note which feature correlates most strongly with price — it'll be relevant when we engineer new features in Section 2.

In [ ]:
# Download, load, and clean the dataset — all in one step!
df = load_dataset()

In [ ]:
# Show 4 charts to explore the data before building the model
plot_dataset(df)

---

# 🔧 2. Data Preparation — With Tricks

This is where the first two tricks come in. Compare with the base notebook:

| Step | Base Notebook | This Notebook |
|------|--------------|---------------|
| Target | Raw price in $ → StandardScaler | 🔧 **Log-transform** of price |
| Features | 4 original features | 🔧 **8 features** (4 original + 4 engineered) |
| Batching | Full dataset each epoch | 🔧 **DataLoader** with mini-batches of 256 |

> 💡 Small changes to *how you represent data* often have a bigger impact on model performance than changes to the architecture itself. This is one of the most important practical lessons in machine learning.

## 🔧 Trick 1: Log-Transform the Target

### The problem with raw prices

In the base notebook we predicted the raw price in dollars and used a StandardScaler on the target. This has a subtle issue: the model optimizes **absolute dollar errors** — an error of \$10,000 counts the same whether the house costs \$50,000 or \$500,000. But in real estate, a \$10k error on a \$50k house (20%) is much worse than on a \$500k house (2%).

### The fix

By applying `log()` to the prices before training, we transform the problem:

$$\text{loss} = (\log(\hat{y}) - \log(y))^2 \approx \left(\frac{\hat{y} - y}{y}\right)^2$$

The model now optimizes **relative (percentage) errors**, which is exactly what we want for prices. To get back to dollars, we simply apply `exp()` to the predictions.

### Why it works here

Look at the price distribution from the plots above — it's right-skewed. The log transform:
- **Compresses** the range (large prices become less extreme)
- **Symmetrizes** the distribution (closer to a Gaussian)
- Makes MSE loss behave like **MAPE** (percentage error)

## 🔧 Trick 2: Feature Engineering — 🎯 Your turn!

### The problem

A neural network with ReLU activations can theoretically approximate any function, but **multiplications between inputs are expensive to learn**. To approximate something as simple as $x_1 \times x_2$, a shallow network needs many neurons and many epochs.

### The fix

We create 4 new features with clear physical/economic meaning:

| New Feature | Formula | Why it matters |
|------------|---------|---------------|
| `room_size` | `square_feet / num_rooms` | Average room size — a proxy for "luxury". Same area with fewer rooms → bigger rooms → higher price |
| `total_space` | `square_feet × num_rooms` | Combined effect: many rooms AND large area → disproportionately expensive |
| `sqft_squared` | `square_feet²` | Price per sqft is not constant — larger houses have a superlinear price increase |
| `dist_squared` | `distance_to_city²` | Distance effect saturates: 1km→5km matters a lot, 40km→44km barely matters |

> ⚠️ **Why not more features?** Adding too many engineered features risks introducing noise and multicollinearity. We pick **few, motivated features** based on domain knowledge, not by blindly generating all possible combinations.

---

### 🎯 Task 1 — Complete the three feature formulas

In the code cell below, `room_size` is already done as an example. Fill in the remaining three using the formulas in the table above.

In [ ]:
# ============================================
# 🔧 TRICK 2: FEATURE ENGINEERING
# ============================================

# Create new features with physical/economic meaning
df['room_size']    = df['square_feet'] / df['num_rooms']       # Average sqft per room
df['total_space']  = df['square_feet'] * df['num_rooms']       # Total usable space
df['sqft_squared'] = df['square_feet'] ** 2                    # Superlinear price effect
df['dist_squared'] = df['distance_to_city(km)'] ** 2           # Saturating distance effect

print(f"✅ Feature engineering complete!")
print(f"   Original features:    4")
print(f"   Engineered features:  4")
print(f"   Total features:       {len(df.columns) - 1}")  # -1 for 'price'
print(f"\n   New columns: {['room_size', 'total_space', 'sqft_squared', 'dist_squared']}")

### 🎯 Task 2 — Compute the log-transformed prices

After you've finished the feature engineering above, fill in `log_prices` in the next cell by applying a logarithm to the price column.

> 💡 Hint: make sure the resulting array has shape `(N, 1)` to match the other target arrays.

The rest (split, scale, tensor conversion) is handled automatically by the helper functions.

In [ ]:
# ============================================
# SEPARATE FEATURES AND TARGET (with log-transform)
# ============================================

# Feature columns — everything except 'price'
feature_columns = [c for c in df.columns if c != 'price']

features = df[feature_columns].values                        # shape: (N, 8) — now 8 features!

# 🔧 TRICK 1: Log-transform the prices
log_prices = np.log(df['price'].values).reshape(-1, 1)       # shape: (N, 1)

# Keep original prices for final evaluation (to compute metrics in dollars)
original_prices = df['price'].values.reshape(-1, 1)

print(f"Number of samples:  {len(log_prices)}")
print(f"Number of features: {features.shape[1]}")
print(f"Feature names:      {feature_columns}")
print(f"\n📊 Price statistics:")
print(f"   Original range:  ${original_prices.min():,.0f} - ${original_prices.max():,.0f}")
print(f"   Log price range: {log_prices.min():.2f} - {log_prices.max():.2f}")

In [ ]:
# Split 80/10/10 and scale the features — all in one step
(train_features, val_features, test_features,
 train_log_prices, val_log_prices, test_log_prices,
 val_orig_prices, test_orig_prices) = split_and_scale_data_tricks(
    features, log_prices, original_prices
)

### 2d. Convert to PyTorch tensors + DataLoader — 🎯 Your turn!

## 🔧 Trick 3: Mini-Batch Training with DataLoader

### The problem with full-batch

In the base notebook, every epoch processes the **entire training set at once**. This means:
- The gradient is computed on all ~8000 samples — very stable but slow to converge
- The model sees the same "big picture" every step — no exploration of the loss landscape
- Memory usage scales linearly with dataset size (problematic for large datasets)

### The fix

We wrap our tensors in a `DataLoader` with `batch_size=256`. Each epoch now processes the data in **mini-batches** of 256 samples:

```
Epoch 1:  batch_1 (256) → update → batch_2 (256) → update → ... → batch_31 (256) → update
Epoch 2:  [shuffled] batch_1 → update → batch_2 → update → ...
```

### Why it works
- **Noisy gradients act as regularization** — the model doesn't overfit to the full-batch gradient direction
- **More weight updates per epoch** — ~31 updates instead of 1, each on a fresh random sample
- **Shuffling** ensures different mini-batches each epoch, exposing the model to varied data orderings
- **Scales to any dataset size** — memory usage depends on batch size, not dataset size

---

### 🎯 Task 3 — Create a TensorDataset and a DataLoader

In the code cell below, the tensor conversion is done for you. Your task is to fill in the two lines that create:
1. `train_dataset` — combine the feature and target tensors into a single dataset object
2. `train_loader` — wrap the dataset so it can be iterated in mini-batches

> 💡 Hints:
> - `TensorDataset` takes tensors as arguments and groups them together.
> - `DataLoader` needs the dataset, the batch size (see the hyperparameter table above), and a flag to shuffle the data each epoch.

In [ ]:
# Convert our data arrays to PyTorch tensors  (already done for you)
(train_features_t, val_features_t, test_features_t,
 train_log_prices_t, val_log_prices_t, test_log_prices_t) = convert_to_tensors(
    train_features, val_features, test_features,
    train_log_prices, val_log_prices, test_log_prices
)

# 🔧 TRICK 3: Create DataLoader for mini-batch training
train_dataset = TensorDataset(train_features_t, train_log_prices_t)
train_loader  = DataLoader(train_dataset, batch_size=256, shuffle=True)

print(f"DataLoader: {len(train_loader)} batches of 256 samples per epoch")

---

# 🧠 3. Neural Network Architecture

Same architecture as the base notebook (`128 → 128 → 64 → 1`), but with two changes:
- **8 input features** instead of 4 (due to feature engineering)
- **Kaiming weight initialization** (Trick 5)

## 🔧 Trick 5: Kaiming (He) Weight Initialization

### The problem with default initialization

PyTorch initializes weights using a uniform distribution that doesn't account for the activation function. With ReLU, which zeros out all negative values, this can lead to:
- **Dead neurons** — some neurons always output 0 and never recover
- **Vanishing/exploding activations** — values shrink or grow as they pass through layers

### The fix

**Kaiming initialization** (He et al., 2015) sets the weight variance to:

$$\text{Var}(w) = \frac{2}{n_{\text{in}}}$$

where $n_{\text{in}}$ is the number of inputs to the layer. The factor of 2 specifically compensates for ReLU zeroing out half the activations. This keeps the variance of activations roughly constant across layers.

### In practice

We add a simple `_init_weights()` method that applies Kaiming to all `Linear` layers and sets biases to zero:

```python
nn.init.kaiming_normal_(layer.weight, nonlinearity='relu')
nn.init.zeros_(layer.bias)
```

In [ ]:
# ============================================
# DEFINE THE NEURAL NETWORK MODEL (with Kaiming init)
# ============================================

class HousePriceModel(nn.Module):
    """
    A neural network for house price prediction.

    Architecture:
        Input → 128 → 128 → 64 → 1
        with ReLU activations and Kaiming initialization.
    """

    def __init__(self, n_features):
        super().__init__()

        self.network = nn.Sequential(
            # Layer 1: input features → 128 neurons
            nn.Linear(n_features, 128),
            nn.ReLU(),

            # Layer 2: 128 → 128 neurons
            nn.Linear(128, 128),
            nn.ReLU(),

            # Layer 3: 128 → 64 neurons
            nn.Linear(128, 64),
            nn.ReLU(),

            # Output layer: 64 → 1 (predicted log price)
            nn.Linear(64, 1)
        )

        # 🔧 TRICK 5: Apply Kaiming initialization
        self._init_weights()

    def _init_weights(self):
        """Apply Kaiming (He) initialization to all Linear layers."""
        for module in self.network:
            if isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(module.weight, nonlinearity='relu')
                nn.init.zeros_(module.bias)

    def forward(self, x):
        """Forward pass: push input x through all layers."""
        return self.network(x)


# Instantiate the model
model = HousePriceModel(
    n_features=train_features_t.shape[1]  # 8 features (4 original + 4 engineered)
)

print(model)
print(f"\n📊 Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"   (vs base notebook: input layer is now 8→128 instead of 4→128)")

---

# 🏋️ 4. Training the Model — With Tricks — 🎯 Your scheduler needed!

Two more changes compared to the base notebook:
- **Mini-batch training** — iterate over the DataLoader instead of the full dataset
- **Learning Rate Scheduler** — automatically reduce LR when the model plateaus

We also keep **early stopping** from the base notebook.

> 💡 **Early stopping** stops training when validation performance stops improving for a set number of epochs ('patience') to reduce overfitting and save time. The best model is typically the checkpoint with the best validation metric.

## 🔧 Trick 4: Learning Rate Scheduler (ReduceLROnPlateau)

### The problem with a fixed learning rate

A fixed LR is a compromise:
- **Too high** → the model oscillates around the minimum and never converges precisely
- **Too low** → training is painfully slow

With a fixed LR, you're stuck with one speed for the entire journey — a compromise between "fast but imprecise" and "slow but accurate".

### The fix

`ReduceLROnPlateau` monitors the validation loss. When it stops improving for `patience` epochs, the scheduler **divides the LR by a factor** (we use 0.5):

```
Early training:  LR = 0.001    (big steps, fast progress)
After plateau:   LR = 0.0005   (smaller steps, fine-tuning)
Later:           LR = 0.00025  (even smaller, precision)
...
```

This gives us the best of both worlds: fast exploration early on, precise convergence later.

### Hyperparameters

| Hyperparameter | Value | Meaning |
|---------------|-------|---------|
| `learning_rate` | 0.001 | Initial step size |
| `weight_decay` | 1e-4 | L2 regularization |
| `max_epochs` | 2000 | Maximum training loops |
| `batch_size` | 256 | 🔧 Samples per mini-batch |
| `scheduler factor` | 0.5 | 🔧 LR multiplier on plateau |
| `scheduler patience` | 50 | 🔧 Epochs to wait before reducing LR |

---

### 🎯 Task 4 — Complete the scheduler configuration

In the next cell, the loss function, optimizer, and number of epochs are already set up. Your task is to fill in the three missing arguments of `ReduceLROnPlateau`: `optimizer`, `mode`, `factor`, and `patience`. Use the values in the table above.

In [ ]:
# ============================================
# SET UP LOSS, OPTIMIZER, AND SCHEDULER
# ============================================

# MSELoss on log-prices  (already done)
loss_function = nn.MSELoss()

# Learning rate and weight decay  (already done)
learning_rate = 0.001
weight_decay = 1e-4

# Adam optimizer  (already done)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay
)

# 🔧 TRICK 4: Learning rate scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',       # Minimize validation loss
    factor=0.5,       # Reduce LR by half
    patience=50       # Wait 50 epochs before reducing
)

number_of_epochs = 2000

print(f"Loss function: {loss_function}")
print(f"Optimizer:     Adam (lr={learning_rate}, weight_decay={weight_decay})")
print(f"🔧 Scheduler:  ReduceLROnPlateau (factor=0.5, patience=50)")
print(f"Max epochs:    {number_of_epochs}")

### The training loop (mini-batch + scheduler)

Each epoch, `run_epoch_minibatch()` handles the boilerplate for us:
- **Inner loop** over `train_loader` batches: forward pass, loss, backward, optimizer step — repeated for each mini-batch
- **Validation** on the full validation set in one pass

Both steps return **float** loss values ready for the scheduler and the early stopping check.

Key differences from the base notebook (highlighted with 🔧):
- `run_epoch_minibatch()` replaces the manual inner batch loop and validation phase 🔧
- **Scheduler step** after each epoch based on validation loss 🔧
- **Learning rate logging** to see when the scheduler kicks in

In [ ]:
# ============================================
# TRAINING LOOP (Mini-batch + Scheduler + Early Stopping)
# ============================================

train_losses = []
val_losses   = []

# Early stopping setup
best_val_loss = float('inf')
patience_counter = 0
patience = 100
best_model_state = None

for epoch in range(number_of_epochs):

    # 🔧 TRICK 3: Train on mini-batches, evaluate on the full validation set
    train_loss, val_loss = run_epoch_minibatch(
        model, optimizer, loss_function,
        train_loader, val_features_t, val_log_prices_t
    )

    # 🔧 TRICK 4: Update learning rate scheduler
    scheduler.step(val_loss)

    # Store losses
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    # --- EARLY STOPPING CHECK ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print(f"\n⏹️  Early stopping triggered at epoch {epoch}")
        print(f"   No improvement for {patience} epochs")
        break

    # Print progress every 100 epochs
    if epoch % 100 == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(
            f"Epoch {epoch:>4d} | "
            f"Train: {train_loss:.4f} | "
            f"Val: {val_loss:.4f} | "
            f"LR: {current_lr:.6f}"
        )

# Load the best model state
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"\n✅ Loaded best model from epoch with val_loss = {best_val_loss:.4f}")

print(f"\n✅ Training complete!")
print(f"   Total epochs run:      {epoch + 1}")
print(f"   Best validation loss:  {best_val_loss:.4f}")

### Visualizing the training history

Same two plots as the base notebook. With mini-batch training and a scheduler, you should notice:
- The curves may be **noisier** than in the base notebook (normal — mini-batches introduce variance in the gradient)
- The loss may drop in **steps** as the scheduler reduces the learning rate at plateaus
- Early stopping usually triggers **later** because mini-batch noise keeps producing small improvements

In [ ]:
# Plot how the training and validation loss changed over time
plot_training_history(train_losses, val_losses)

---

# 📏 5. Model Evaluation

Same metrics as the base notebook. The key difference is that our model outputs **log-prices**, so `get_predictions_log()` applies `np.exp()` to convert predictions back to dollars before computing metrics.

**Compare these numbers with the base notebook** to see the combined impact of the 5 tricks. Pay particular attention to MAPE — the log-transform (Trick 1) is specifically designed to make percentage errors more uniform across all price ranges.

In [ ]:
# Run the model on the test set and convert log-price predictions back to real dollars
test_predictions = get_predictions_log(model, test_features_t)

In [ ]:
# Print MAE, RMSE, MAPE, and R² — overall and split by price range
print_performance_metrics(test_orig_prices, test_predictions, show_r2=True)

print("\n📝 Compare these with the base notebook to see the improvement!")

---

# 📊 6. Prediction Visualizations

Same three plots as the base notebook. After applying the 5 tricks, look for these improvements:

1. **Predicted vs Actual** — points should cluster more tightly around the diagonal
2. **Residuals histogram** — the distribution should be narrower and more centered at 0
3. **Sample comparison** — the orange and blue bars should be closer together

In [ ]:
# Show 3 charts comparing what the model predicted vs the actual prices
plot_predictions(test_orig_prices, test_predictions)

---

## 🎯 Summary: What We Changed and Why

| # | Trick | Where it acts | Expected improvement |
|---|-------|--------------|---------------------|
| 🔧 1 | **Log-transform target** | Preprocessing | Lower MAPE, more uniform errors across price ranges |
| 🔧 2 | **Feature engineering** | Preprocessing | Lower MAE/RMSE, captures non-linear relationships |
| 🔧 3 | **Mini-batch DataLoader** | Training loop | Better generalization, more updates per epoch |
| 🔧 4 | **LR Scheduler** | Training loop | Fine-tuned convergence, lower final loss |
| 🔧 5 | **Kaiming initialization** | Model definition | Faster initial convergence, fewer dead neurons |

### Key takeaways for students

1. **Data preparation often matters more than architecture** — Tricks 1 and 2 (log-transform and feature engineering) are likely the biggest contributors to improvement.
2. **Training dynamics matter** — Tricks 3 and 4 (DataLoader and scheduler) don't change *what* the model learns, but *how efficiently* it learns.
3. **Small details add up** — Trick 5 (Kaiming init) is a one-liner, but combined with the others, it contributes to a cleaner training trajectory.
4. **Always compare** — Without the base notebook as a reference, we wouldn't know which tricks actually help. Controlled experiments are essential.